# Convolutional Neural Networks (CNNs)

Regular neural networks treat every pixel as an independent input. A 28×28 grayscale image = 784 inputs, all connected to every neuron in the first layer. For a 224×224 color image, that's 150,528 inputs — and one hidden layer of 1000 neurons would need **150 million** weights. That's wasteful and doesn't capture spatial structure.

**CNNs exploit two insights:**
1. **Local patterns** — an edge is an edge regardless of where it appears
2. **Hierarchy** — simple features (edges) combine into complex ones (eyes → faces)

The result: far fewer parameters, spatial awareness, and state-of-the-art image recognition.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 1. The Convolution Operation

A **filter** (kernel) slides across the image, computing a dot product at each position. Each filter detects a specific pattern: horizontal edge, vertical edge, corner, texture.

```
Image (5×5)          Filter (3×3)         Output (3×3)
┌─────────────┐      ┌─────────┐          ┌─────────┐
│ 1 0 1 0 1   │      │ 1 0 1   │          │ 4 3 4   │
│ 0 1 0 1 0   │  ✱   │ 0 1 0   │    =     │ 2 4 2   │
│ 1 0 1 0 1   │      │ 1 0 1   │          │ 4 3 4   │
│ 0 1 0 1 0   │      └─────────┘          └─────────┘
│ 1 0 1 0 1   │
└─────────────┘
```

Output size: `(input_size - filter_size) / stride + 1`

In [ ]:
image = np.array([
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1]
], dtype=float)

vertical_edge = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=float)

horizontal_edge = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
], dtype=float)

def convolve2d(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(patch * kernel)
    return output

vert_output = convolve2d(image, vertical_edge)
horiz_output = convolve2d(image, horizontal_edge)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
titles = ['Input Image', 'Vertical Edge Filter', 'Vertical Edge Detected', 'Horizontal Edge Detected']
images = [image, vertical_edge, vert_output, horiz_output]
cmaps = ['gray', 'RdBu', 'RdBu', 'RdBu']

for ax, img, title, cmap in zip(axes, images, titles, cmaps):
    im = ax.imshow(img, cmap=cmap, interpolation='nearest')
    ax.set_title(title, fontweight='bold', fontsize=11)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            ax.text(j, i, f'{img[i,j]:.0f}', ha='center', va='center', fontsize=10,
                   color='white' if abs(img[i,j]) > 1 else 'black')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()
print("The vertical filter detects the left-to-right transition (the vertical edge).")
print("The horizontal filter finds nothing because there are no top-to-bottom transitions.")

---
## 2. Pooling — Shrink While Keeping the Signal

After convolution, we reduce spatial size with **pooling**:
- **Max pooling** — takes the maximum value in each region (most common)
- **Average pooling** — takes the mean

Benefits: reduces computation, provides translational invariance, prevents overfitting.

In [ ]:
feature_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [3, 2, 8, 7],
    [4, 1, 5, 6]
], dtype=float)

def max_pool_2d(x, size=2):
    h, w = x.shape
    out = np.zeros((h // size, w // size))
    for i in range(0, h, size):
        for j in range(0, w, size):
            out[i // size, j // size] = np.max(x[i:i+size, j:j+size])
    return out

def avg_pool_2d(x, size=2):
    h, w = x.shape
    out = np.zeros((h // size, w // size))
    for i in range(0, h, size):
        for j in range(0, w, size):
            out[i // size, j // size] = np.mean(x[i:i+size, j:j+size])
    return out

max_pooled = max_pool_2d(feature_map)
avg_pooled = avg_pool_2d(feature_map)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, data, title in zip(axes,
                            [feature_map, max_pooled, avg_pooled],
                            ['Feature Map (4×4)', 'Max Pool 2×2 → (2×2)', 'Avg Pool 2×2 → (2×2)']):
    ax.imshow(data, cmap='YlOrRd', interpolation='nearest')
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i,j]:.0f}', ha='center', va='center', fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

---
## 3. CNN Architecture — The Full Pipeline

```
Input Image
    │
    ▼
┌──────────┐    ┌──────────┐    ┌──────────┐
│  Conv2d  │───►│  ReLU    │───►│ MaxPool  │  ← Feature extraction (repeat)
└──────────┘    └──────────┘    └──────────┘
    │
    ▼
┌──────────┐    ┌──────────┐    ┌──────────┐
│  Conv2d  │───►│  ReLU    │───►│ MaxPool  │  ← Deeper features
└──────────┘    └──────────┘    └──────────┘
    │
    ▼
┌──────────┐    ┌──────────┐    ┌──────────┐
│ Flatten  │───►│ Linear   │───►│ Softmax  │  ← Classification
└──────────┘    └──────────┘    └──────────┘
```

**Conv layers** extract spatial features. **Pooling** reduces spatial size. **Flatten + Linear** does the classification.

---
## 4. Build a CNN in PyTorch

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28×28 → 28×28, 16 filters
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 28×28 → 14×14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 14×14 → 14×14, 32 filters
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 14×14 → 7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                  # 32×7×7 = 1568
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN()
print(model)

dummy = torch.randn(1, 1, 28, 28)
print(f"\nInput: {dummy.shape}")
print(f"Output: {model(dummy).shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 5. Train on MNIST

MNIST: 70,000 handwritten digits (28×28 grayscale). The "Hello World" of deep learning.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Image shape:      {train_dataset[0][0].shape}")
print(f"Classes:          0-9")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label))
    ax.axis('off')
plt.suptitle('MNIST Samples', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
torch.manual_seed(42)
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses, test_accs = [], []

for epoch in range(5):
    model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        output = model(X_batch)
        loss = criterion(output, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    train_losses.append(running_loss / len(train_loader))
    
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            _, predicted = output.max(1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    
    acc = correct / total
    test_accs.append(acc)
    print(f"  Epoch {epoch+1}/5 | Loss: {train_losses[-1]:.4f} | Test Acc: {acc:.4f}")

print(f"\nFinal accuracy: {test_accs[-1]:.2%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, 'o-', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot([a * 100 for a in test_accs], 'o-', color='#2ecc71', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Test Accuracy', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(95, 100)

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
test_images, test_labels = [], []
for img, label in test_dataset:
    test_images.append(img)
    test_labels.append(label)
    if len(test_images) == 16:
        break

batch = torch.stack(test_images).to(device)
with torch.no_grad():
    preds = model(batch).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(test_images[i].squeeze(), cmap='gray')
    color = '#2ecc71' if preds[i] == test_labels[i] else '#e74c3c'
    ax.set_title(f'Pred: {preds[i].item()}', color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('CNN Predictions (green=correct, red=wrong)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Train on CIFAR-10 — A Harder Challenge

CIFAR-10: 60,000 color images (32×32×3) across 10 classes. Much harder than MNIST — color, varied backgrounds, more complex shapes.

In [ ]:
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)

cifar_train_loader = DataLoader(cifar_train, batch_size=64, shuffle=True)
cifar_test_loader = DataLoader(cifar_test, batch_size=256, shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = cifar_train[i]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    ax.set_title(classes[label], fontsize=9)
    ax.axis('off')
plt.suptitle('CIFAR-10 Samples', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),    # 32×32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                     # → 16×16
            nn.Dropout2d(0.25),
            
            nn.Conv2d(32, 64, 3, padding=1),    # 16×16
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                     # → 8×8
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10),
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(42)
cifar_model = CIFAR_CNN().to(device)
print(f"Parameters: {sum(p.numel() for p in cifar_model.parameters()):,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cifar_model.parameters(), lr=1e-3)

cifar_train_losses, cifar_test_accs = [], []

for epoch in range(10):
    cifar_model.train()
    running_loss = 0
    for X_batch, y_batch in cifar_train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        output = cifar_model(X_batch)
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    cifar_train_losses.append(running_loss / len(cifar_train_loader))
    
    cifar_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in cifar_test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = cifar_model(X_batch).argmax(1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    
    acc = correct / total
    cifar_test_accs.append(acc)
    print(f"  Epoch {epoch+1:2d}/10 | Loss: {cifar_train_losses[-1]:.4f} | Test Acc: {acc:.4f}")

print(f"\nFinal CIFAR-10 accuracy: {cifar_test_accs[-1]:.2%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(cifar_train_losses, 'o-', color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('CIFAR-10 Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot([a * 100 for a in cifar_test_accs], 'o-', color='#2ecc71', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('CIFAR-10 Test Accuracy', fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Visualize What Filters Learn

The first conv layer learns simple patterns: edge detectors at different orientations. Deeper layers combine these into increasingly abstract features.

In [ ]:
first_conv = list(model.features.children())[0]
filters = first_conv.weight.data.cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i, 0], cmap='RdBu', interpolation='nearest')
    ax.set_title(f'Filter {i}', fontsize=9)
    ax.axis('off')
plt.suptitle('Learned Filters from First Conv Layer (MNIST CNN)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
sample_img = test_dataset[0][0].unsqueeze(0).to(device)

model.eval()
activations = []
x = sample_img
for layer in model.features:
    x = layer(x)
    if isinstance(layer, nn.Conv2d):
        activations.append(x.detach().cpu())

fig, axes = plt.subplots(2, 9, figsize=(16, 4))

axes[0, 0].imshow(sample_img.cpu().squeeze(), cmap='gray')
axes[0, 0].set_title('Input', fontweight='bold')
axes[0, 0].axis('off')

for i in range(8):
    axes[0, i+1].imshow(activations[0][0, i], cmap='viridis')
    axes[0, i+1].set_title(f'L1 F{i}', fontsize=8)
    axes[0, i+1].axis('off')

axes[1, 0].axis('off')
for i in range(8):
    axes[1, i+1].imshow(activations[1][0, i], cmap='viridis')
    axes[1, i+1].set_title(f'L2 F{i}', fontsize=8)
    axes[1, i+1].axis('off')

plt.suptitle('Feature Maps at Different Layers', fontweight='bold')
plt.tight_layout()
plt.show()
print("Layer 1 detects edges. Layer 2 combines edges into more complex patterns.")

---
## 8. Data Augmentation

Data augmentation artificially expands the training set by applying random transformations. The model never sees the exact same image twice — this forces it to learn robust features instead of memorizing specific pixels.

In [ ]:
augment_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

raw_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                                            transform=transforms.ToTensor())
aug_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True,
                                            transform=augment_transform)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))

for i in range(8):
    img_raw, _ = raw_dataset[0]
    axes[0, i].imshow(img_raw.permute(1, 2, 0).numpy())
    axes[0, i].set_title('Original' if i == 0 else '', fontsize=9)
    axes[0, i].axis('off')

for i in range(8):
    img_aug, _ = aug_dataset[0]
    img_np = img_aug.permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.2470, 0.2435, 0.2616]) + np.array([0.4914, 0.4822, 0.4465])
    img_np = np.clip(img_np, 0, 1)
    axes[1, i].imshow(img_np)
    axes[1, i].set_title('Augmented' if i == 0 else '', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Data Augmentation — Same Image, Different Transforms', fontweight='bold')
plt.tight_layout()
plt.show()
print("Each epoch, the model sees a different variation of every image.")
print("This is free regularization — reduces overfitting without collecting more data.")

In [ ]:
print("Common augmentation strategies:")
print("")
print("┌──────────────────┬─────────────────────────────────────────────┐")
print("│ Transform        │ When to Use                                │")
print("├──────────────────┼─────────────────────────────────────────────┤")
print("│ HorizontalFlip   │ Most images (not text/numbers)             │")
print("│ RandomRotation   │ Objects that can appear rotated            │")
print("│ RandomCrop       │ Always useful — forces spatial invariance  │")
print("│ ColorJitter      │ Real-world photos with variable lighting   │")
print("│ RandomErasing    │ Encourages learning multiple features      │")
print("│ GaussianBlur     │ Adds robustness to image quality           │")
print("└──────────────────┴─────────────────────────────────────────────┘")

---
## Key Takeaways

1. **Convolution** = sliding filter that detects local patterns (edges, textures)
2. **Pooling** = downsample to reduce computation and gain translation invariance
3. **CNN architecture** = Conv → ReLU → Pool (repeat) → Flatten → FC → Output
4. **Parameter sharing** = same filter used everywhere → far fewer parameters than dense networks
5. **Feature hierarchy** = early layers learn edges, later layers learn complex patterns
6. **Data augmentation** = free regularization through random transforms
7. **MNIST ~99%** with a simple CNN; **CIFAR-10 ~75-80%** needs deeper architecture + augmentation